# 02 - Finding PWE

Epilepsy was identified using a medication-based case definition. Participants were classified as having epilepsy if they reported current use of one or more antiseizure medications in the NHANES prescription medication inventory.

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import pyreadstat
import matplotlib.pyplot as plt

import ambient_light_epilepsy.nhanes as nhn  
import ambient_light_epilepsy.cohort as ch
import time

## Find people with epilepsy using prescription medication data

In [2]:
year = "G"
pwe_g = ch.find_people_on_asm(year)

CSV file already exists in C:\Users\ahernj\Documents\Projects\ambient_light_epilepsy_analysis\data\G\processed\people_with_epilepsy_G.csv


In [4]:
year = "H"
pwe_h = ch.find_people_on_asm(year)

CSV file already exists in C:\Users\ahernj\Documents\Projects\ambient_light_epilepsy_analysis\data\H\processed\people_with_epilepsy_H.csv


## Analysis of PWE prescription medication

This is not essentially analysis, but is easy enough to do now in case it becomes useful later.

In [3]:
pwe_seqn = pwe.values

rx_pwe = rx[rx["SEQN"].isin(pwe_seqn)].copy()

rx_pwe["drug_lower"] = rx_pwe["RXDDRUG"].str.lower()
rx_pwe["is_epilepsy_med"] = rx_pwe["drug_lower"].isin(epilepsy_specific_asms)

# keep current medications only
rx_pwe = rx_pwe[rx_pwe["RXDUSE"] == 1]


med_counts = (
    rx_pwe
    .groupby(["RXDDRUG", "is_epilepsy_med"])["SEQN"]
    .nunique()
    .reset_index(name="n_users")
    .sort_values("n_users", ascending=False)
)


colors = med_counts["is_epilepsy_med"].map(
    {True: "orange", False: "grey"}
)

plt.figure(figsize=(30, 6))
plt.bar(med_counts["RXDDRUG"], med_counts["n_users"], color=colors)

plt.xticks(rotation=90)
plt.ylabel("Number of participants")
plt.title("Medication use among people with epilepsy (NHANES 2011–2012)")

plt.tight_layout()
plt.show()



NameError: name 'pwe' is not defined

In [ ]:
epilepsy_only = rx_pwe[rx_pwe["is_epilepsy_med"]]

polytherapy_counts = (
    epilepsy_only
    .groupby("SEQN")["drug_lower"]
    .nunique()
    .reset_index(name="n_epilepsy_meds")
)

polytherapy = polytherapy_counts[polytherapy_counts["n_epilepsy_meds"] > 1]


polytherapy_report = (
    epilepsy_only
    .groupby("SEQN")["RXDDRUG"]
    .unique()
    .reset_index()
)

polytherapy_report = polytherapy_report.merge(
    polytherapy_counts, on="SEQN"
)

polytherapy_report = polytherapy_report[
    polytherapy_report["n_epilepsy_meds"] > 1
]


In [ ]:
polytherapy_report

In [ ]:
n_pwe = len(pwe_seqn)
n_polytherapy = polytherapy.shape[0]
pct_polytherapy = 100 * n_polytherapy / n_pwe

print(f"PWE identified: {n_pwe}")
print(f"On >1 epilepsy medication: {n_polytherapy} ({pct_polytherapy:.1f}%)")
